In [7]:
import os
import gc
import time
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import log_loss, roc_auc_score
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
CONFIG = {
    "seed": 42,
    "batch_size": 128,
    "embedding_dim": 32, # Keep small for memory efficiency
    "lr": 1e-3,
    "epochs": 1,         # 1 Epoch is usually enough for Avazu on this scale
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "min_freq": 10,      # Map categories appearing < 10 times to <UNK>
    "num_workers": 2,
    "train_path": "./data/train.gz",
    "test_path": "./data/test.gz",
    "sub_path": "submission.csv"
}

def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CONFIG['seed'])
print(f"Using device: {CONFIG['device']}")

Using device: cuda


In [ ]:
def process_data_polars():
    print("Loading data with Polars (Schema Fixed)...")
    
    # --- 1. Define Schema to prevent Parsing Errors ---
    # We force 'id' to String to handle the massive integers.
    # We force other categoricals to String initially to speed up parsing.
    dtypes = {
        'id': pl.String,
        'click': pl.UInt8,
        'hour': pl.String,
        'C1': pl.String,
        'banner_pos': pl.String,
        'site_id': pl.String,
        'site_domain': pl.String,
        'site_category': pl.String,
        'app_id': pl.String,
        'app_domain': pl.String,
        'app_category': pl.String,
        'device_id': pl.String,
        'device_ip': pl.String,
        'device_model': pl.String,
        'device_type': pl.String,
        'device_conn_type': pl.String,
        'C14': pl.String,
        'C15': pl.String,
        'C16': pl.String,
        'C17': pl.String,
        'C18': pl.String,
        'C19': pl.String,
        'C20': pl.String,
        'C21': pl.String,
    }
    
    # Columns to actually train on
    cat_cols = [
        'C1', 'banner_pos', 'site_id', 'site_domain', 'site_category',
        'app_id', 'app_domain', 'app_category', 'device_id', 'device_ip',
        'device_model', 'device_type', 'device_conn_type',
        'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21'
    ]
    
    # Lazy load with explicit dtypes
    q_train = pl.scan_csv(CONFIG['train_path'], schema_overrides=dtypes)
    q_test = pl.scan_csv(CONFIG['test_path'], schema_overrides=dtypes)
    
    # --- 2. Feature Engineering (Time) ---
    def extract_time_features(q):
        return q.with_columns([
            pl.col("hour").str.slice(6, 2).cast(pl.UInt8).alias("hour_of_day"),
            pl.col("hour").str.slice(4, 2).cast(pl.UInt8).alias("day_of_week"),
        ])

    q_train = extract_time_features(q_train)
    q_test = extract_time_features(q_test)
    
    # Add new time features to the list of categorical columns
    cat_cols += ['hour_of_day', 'day_of_week']

    print("Building vocabularies (Frequency Thresholding)...")
    
    vocab_sizes = {}
    feat_maps = {}
    
    # We iterate to save memory, but with 30GB you could potentially do groups.
    # Iteration is safer to avoid OOM spikes during the 'collect'.
    for col in tqdm(cat_cols):
        # 1. Count frequencies in Train
        # We cast to String to ensure matching types between Train/Test
        counts = q_train.select(pl.col(col).cast(pl.String)).group_by(col).len().collect()
        
        # 2. Filter: Keep only features appearing >= min_freq
        frequent_items = counts.filter(pl.col("len") >= CONFIG['min_freq'])[col].to_list()
        
        # 3. Create Map: Value -> Int ID (Start at 1, 0 is <UNK>)
        # Using a dictionary is fast for Polars 'replace'
        mapping = {val: i + 1 for i, val in enumerate(frequent_items)}
        
        feat_maps[col] = mapping
        vocab_sizes[col] = len(mapping) + 1
        
    print("Vocabularies built. Mapping and converting to Numpy...")

    # --- 3. Transformation Function ---
    def map_and_convert(df_lazy, is_test=False):
        # Select only necessary columns to save RAM
        cols_to_select = cat_cols + (['id'] if is_test else ['click'])
        
        # Materialize the dataframe now. 
        # With 30GB RAM, we can load the full dataset (subset of columns) into memory.
        df = df_lazy.select(cols_to_select).collect()
        
        # Extract ID or Target before mapping
        extra_data = df['id'].to_numpy() if is_test else df['click'].to_numpy().astype(np.float32)
        
        # Perform mapping in-place (or close to it)
        # We loop through columns to map them to integers
        for col in tqdm(cat_cols, desc=f"Mapping {'Test' if is_test else 'Train'}"):
            mapping = feat_maps[col]
            
            # Polars 'replace' is very optimized. 
            # Values not in the mapping (rare or new) become null, we fill with 0 (<UNK>)
            df = df.with_columns(
                pl.col(col).cast(pl.String) # Ensure type match
                .replace(mapping, default=0)
                .cast(pl.Int32)
            )
        
        # Convert the feature matrix to Numpy
        X_data = df.select(cat_cols).to_numpy().astype(np.int32)
        
        # Clean up Polars DF to free RAM immediately
        del df
        gc.collect()
        
        return X_data, extra_data

    # Process Train
    X_train, y_train = map_and_convert(q_train, is_test=False)
    print(f"Train processed. Shape: {X_train.shape}")
    
    # Process Test
    X_test, test_ids = map_and_convert(q_test, is_test=True)
    print(f"Test processed. Shape: {X_test.shape}")
    
    return X_train, y_train, X_test, test_ids, vocab_sizes, cat_cols

# Run the processing
X_train, y_train, X_test, test_ids, vocab_sizes, feature_names = process_data_polars()

Loading data with Polars (Schema Fixed)...
Building vocabularies (Frequency Thresholding)...


  0%|          | 0/23 [00:00<?, ?it/s]

In [ ]:
class AvazuDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

# Create DataLoaders
train_dataset = AvazuDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True)

test_dataset = AvazuDataset(X_test)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)

del X_train, y_train, X_test
gc.collect()

In [ ]:
class FeatureGatingLayer(nn.Module):
    """
    The 'Fast' implementation of the Gated Attention paper.
    Instead of O(N^2) Self-Attention, we use O(N) Element-wise Gating.
    It learns to suppress noise (sparsity) and adds non-linearity.
    """
    def __init__(self, input_dim):
        super().__init__()
        self.gate_linear = nn.Linear(input_dim, input_dim)
        # self.norm = nn.LayerNorm(input_dim) # Helps stability

    def forward(self, x):
        # x shape: [Batch, Num_Features * Embed_Dim]
        
        # Calculate Gate Score (Sigmoid forces 0-1 range)
        gate_score = torch.sigmoid(self.gate_linear(x))
        
        # Apply Gate
        return x * gate_score

class DCNv2(nn.Module):
    """
    Deep Cross Network V2 (Parallel).
    Explicitly captures high-order feature interactions.
    """
    def __init__(self, input_dim, num_layers=2):
        super().__init__()
        self.num_layers = num_layers
        self.input_dim = input_dim
        
        # Parameters for Cross Layers
        self.W = nn.ParameterList([nn.Parameter(torch.randn(input_dim, input_dim)) for _ in range(num_layers)])
        self.b = nn.ParameterList([nn.Parameter(torch.zeros(input_dim)) for _ in range(num_layers)])
        
        # Init
        for w in self.W:
            nn.init.xavier_uniform_(w)

    def forward(self, x):
        # x: [Batch, Input_Dim]
        x0 = x
        xi = x
        
        for i in range(self.num_layers):
            # x_next = x0 * (W * xi + b) + xi
            # We use linear layer logic for W * xi + b
            feature_crossing = torch.matmul(xi, self.W[i]) + self.b[i]
            xi = x0 * feature_crossing + xi
            
        return xi

class GatedDCNModel(nn.Module):
    def __init__(self, vocab_sizes, embedding_dim, feature_names):
        super().__init__()
        self.feature_names = feature_names
        
        # 1. Embedding Layer
        self.embeddings = nn.ModuleDict()
        total_dim = 0
        for feat in feature_names:
            self.embeddings[feat] = nn.Embedding(vocab_sizes[feat], embedding_dim)
            total_dim += embedding_dim
            
        # 2. Feature Gating (Replaces SENet)
        self.gating = FeatureGatingLayer(total_dim)
        
        # 3. DCNv2
        self.dcn = DCNv2(total_dim, num_layers=2)
        
        # 4. Final MLP
        self.mlp = nn.Sequential(
            nn.Linear(total_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        # x shape: [Batch, Num_Features]
        
        # Flatten inputs into a single dense vector
        embeds = []
        for i, feat in enumerate(self.feature_names):
            embeds.append(self.embeddings[feat](x[:, i]))
        
        # Concatenate: [Batch, Total_Dim]
        dnn_input = torch.cat(embeds, dim=1)
        
        # Apply Gating (Sparsity & Reweighting)
        gated_input = self.gating(dnn_input)
        
        # Apply Cross Network (Interactions)
        cross_out = self.dcn(gated_input)
        
        # Final Prediction
        logits = self.mlp(cross_out)
        return torch.sigmoid(logits)

# Initialize Model
model = GatedDCNModel(vocab_sizes, CONFIG['embedding_dim'], feature_names)
model.to(CONFIG['device'])
print(model)

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-5)

print("Starting Training...")

for epoch in range(CONFIG['epochs']):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for X_batch, y_batch in pbar:
        X_batch, y_batch = X_batch.to(CONFIG['device']), y_batch.to(CONFIG['device']).unsqueeze(1)
        
        optimizer.zero_grad()
        
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Done. Avg Loss: {avg_loss:.5f}. Time: {time.time() - start_time:.0f}s")

In [ ]:
print("Starting Inference...")
model.eval()
predictions = []

with torch.no_grad():
    for X_batch in tqdm(test_loader, desc="Predicting"):
        X_batch = X_batch.to(CONFIG['device'])
        
        preds = model(X_batch)
            
        predictions.append(preds.cpu().numpy())

# Concatenate predictions
predictions = np.concatenate(predictions).flatten()

print("Creating submission file...")
submission = pl.DataFrame({
    "id": test_ids,
    "click": predictions
})

submission.write_csv(CONFIG['sub_path'])
print(f"Submission saved to {CONFIG['sub_path']}")
print(submission.head())